In [1]:
from pathlib import Path
from collections import Counter

from qonnx.core.modelwrapper import ModelWrapper
from qonnx.custom_op.registry import getCustomOp


MODEL_PATHS = [
    Path("/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_convert_to_hw.onnx"),
    Path("/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_specialize_layers.onnx"),
    Path("/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_apply_folding_config.onnx"),
]


def safe_shape(model, name):
    try:
        return model.get_tensor_shape(name)
    except Exception as e:
        return f"<shape error: {e}>"


def safe_dtype(model, name):
    try:
        return model.get_tensor_datatype(name)
    except Exception as e:
        return f"<dtype error: {e}>"


def print_node_attrs(node):
    try:
        cop = getCustomOp(node)
    except Exception:
        return

    attrs = [
        "MW",
        "MH",
        "PE",
        "SIMD",
        "noActivation",
        "inputDataType",
        "weightDataType",
        "outputDataType",
        "accDataType",
        "actVal",
        "numInputVectors",
    ]

    for a in attrs:
        try:
            print(f"      {a}: {cop.get_nodeattr(a)}")
        except Exception:
            pass


for path in MODEL_PATHS:
    if not path.exists():
        print("\n[MISSING]", path)
        continue

    print("\n\n===================================================")
    print("MODEL:", path)
    print("===================================================")

    model = ModelWrapper(str(path))

    ops = Counter([n.op_type for n in model.graph.node])
    print("\nOp histogram:")
    for k, v in sorted(ops.items()):
        print(f"  {k}: {v}")

    print("\nGraph outputs:")
    for out in model.graph.output:
        name = out.name
        print("  output:", name)
        print("    shape:", safe_shape(model, name))
        print("    dtype:", safe_dtype(model, name))

        prod = model.find_producer(name)
        if prod is None:
            print("    producer: None")
        else:
            print("    producer:", prod.name, prod.op_type, prod.domain)
            print_node_attrs(prod)

    print("\nLast 20 nodes:")
    for i, node in enumerate(model.graph.node[-20:]):
        print(f"\n[{len(model.graph.node)-20+i}] {node.name} | {node.op_type} | {node.domain}")
        print("  inputs :", list(node.input))
        print("  outputs:", list(node.output))

        for t in list(node.input) + list(node.output):
            print("   ", t, "shape=", safe_shape(model, t), "dtype=", safe_dtype(model, t))

        if "MVAU" in node.op_type or "Threshold" in node.op_type or node.op_type in ["MultiThreshold", "Quant"]:
            print_node_attrs(node)

    print("\nMVAU nodes with MH=5 or output class-like:")
    for node in model.graph.node:
        if "MVAU" not in node.op_type:
            continue

        try:
            cop = getCustomOp(node)
            mh = cop.get_nodeattr("MH")
        except Exception:
            mh = None

        if mh == 5:
            print("\n  node:", node.name, node.op_type)
            print("  outputs:", list(node.output))
            print_node_attrs(node)

            for t in list(node.output):
                print("    out tensor:", t, "shape=", safe_shape(model, t), "dtype=", safe_dtype(model, t))
                consumers = model.find_consumers(t)
                if consumers is not None:
                    print("    consumers:", [(c.name, c.op_type) for c in consumers])



MODEL: /home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_convert_to_hw.onnx

Op histogram:
  Add: 4
  ConvolutionInputGenerator: 6
  MVAU: 9
  MatMul: 1
  Mul: 7
  MultiThreshold: 7
  Pad: 6
  Thresholding: 4
  Transpose: 23
  Unsqueeze: 1

Graph outputs:
  output: global_out
    shape: [1, 5]
    dtype: FLOAT32
    producer: Add_3 Add 

Last 20 nodes:

[48] Pad_4 | Pad | 
  inputs : ['Mul_4_out0', 'Pad_4_param0', 'Pad_4_param1']
  outputs: ['Pad_4_out0']
    Mul_4_out0 shape= [1, 1, 1, 140] dtype= FLOAT32
    Pad_4_param0 shape= [8] dtype= FLOAT32
    Pad_4_param1 shape= [] dtype= FLOAT32
    Pad_4_out0 shape= [1, 1, 1, 148] dtype= FLOAT32

[49] MultiThreshold_5 | MultiThreshold | qonnx.custom_op.general
  inputs : ['Pad_4_out0', 'MultiThreshold_13_param0']
  outputs: ['MultiThreshold_13_out0']
    Pad_4_out0 shape= [1, 1, 1, 148] dtype= FLOAT32
    MultiThreshold_13_param0 shape= [1, 255] dtype= FLOAT32
    MultiThreshold_13_

In [2]:
from pathlib import Path
import numpy as np
from qonnx.core.modelwrapper import ModelWrapper
import qonnx.core.onnx_exec as oxe

MODEL_PATH = Path(
    "/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/"
    "finn_build_pynqz2/intermediate_models/step_apply_folding_config.onnx"
)

DATA_PATH = Path(
    "/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/"
    "datasets/ECG5000/ECG5000_val_split.txt"
)

NUM_CLASSES = 5
SEQ_LEN = 140

model = ModelWrapper(str(MODEL_PATH))
input_name = model.graph.input[0].name

print("model input:", input_name)
print("input shape:", model.get_tensor_shape(input_name))
print("input dtype:", model.get_tensor_datatype(input_name))

data = np.loadtxt(str(DATA_PATH), dtype=np.float32)
labels = data[:, 0].astype(np.int32) - 1
x_raw = data[:, 1:].astype(np.float32)

def run_one(x_values, tag):
    x = x_values.reshape(1, 140, 1, 1).astype(np.float32)

    out_dict = oxe.execute_onnx(model, {input_name: x})
    print("\n===", tag, "===")

    for name, val in out_dict.items():
        arr = np.asarray(val)
        print(name, arr.shape, arr.dtype)
        print(arr.reshape(-1)[:20])

    out = list(out_dict.values())[0]
    logits = np.asarray(out).reshape(-1)
    print("logits:", logits)
    print("pred:", int(np.argmax(logits)))

x0 = x_raw[0]
print("label:", labels[0])
print("raw first 20:", x0[:20])
print("round raw first 20:", np.round(x0[:20]))
print("round raw*16 first 20:", np.round(x0[:20] * 16))

run_one(x0, "raw float input")
run_one(np.round(x0), "round raw input")
run_one(np.round(x0 * 16), "round raw * 16 input")

model input: global_in
input shape: [1, 140, 1, 1]
input dtype: INT8
label: 0
raw first 20: [ 2.1043675   0.34081385 -2.434759   -3.7213652  -3.996241   -3.9848614
 -3.4466867  -2.2484236  -1.5658237  -1.4222693  -0.7003251  -0.07998953
 -0.13492504 -0.20195971 -0.17599626 -0.1398821  -0.18158206 -0.26193014
 -0.21033768 -0.19628556]
round raw first 20: [ 2.  0. -2. -4. -4. -4. -3. -2. -2. -1. -1. -0. -0. -0. -0. -0. -0. -0.
 -0. -0.]
round raw*16 first 20: [ 34.   5. -39. -60. -64. -64. -55. -36. -25. -23. -11.  -1.  -2.  -3.
  -3.  -2.  -3.  -4.  -3.  -3.]


/home/slowman/Desktop/project/Thesis/finn/deps/qonnx/src/qonnx/util/basic.py:296: UserWarning: The values of tensor global_in can't be represented with the set datatype annotation (INT8), they will be rounded to match the datatype annotation.
  warnings.warn(


Exception: Rounding error is too high to match set QONNX
            datatype (INT8) for input global_in